# 01 - Cleaning + Stratified Split (IndoBERTweet, tanpa resample)

In [1]:
import re
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

BASE_DIR = Path(r"C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data")
LABELED_DIR = BASE_DIR / "labeled"
PRAPROCESSING_DIR = BASE_DIR / "praprocessing"

# ===== UBAH DI SINI kalau mau pakai file lain =====
INPUT_FILENAME = "labels_shafwa_hans_agreement.csv"
TEST_SIZE = 0.20
RANDOM_STATE = 42
# ==================================================

INPUT_PATH = LABELED_DIR / INPUT_FILENAME
if not INPUT_PATH.exists():
    print("File tidak ditemukan. File yang ada di", LABELED_DIR)
    for f in sorted(LABELED_DIR.glob("*.csv")):
        print("  -", f.name)
    raise FileNotFoundError(INPUT_PATH)

# Output: data/praprocessing/<nama_file_tanpa_csv>/
OUTPUT_DIR = PRAPROCESSING_DIR / INPUT_PATH.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input :", INPUT_PATH)
print("Output:", OUTPUT_DIR)

Input : C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data\labeled\labels_shafwa_hans_agreement.csv
Output: C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data\praprocessing\labels_shafwa_hans_agreement


In [2]:
# ===== LOAD =====
df = pd.read_csv(INPUT_PATH, dtype=str, encoding="utf-8")
print("Total baris:", len(df))
print("Kolom:", list(df.columns))
print("\nMissing values:\n", df[["comment_text", "final_label"]].isnull().sum())
print("\nDistribusi label awal:")
print(df["final_label"].value_counts())

Total baris: 1959
Kolom: ['post_url', 'username', 'comment_text', 'timestamp', 'likes_count', 'label_shafwa', 'label_hans', 'votes_positif', 'votes_netral', 'votes_negatif', 'final_label']

Missing values:
 comment_text    51
final_label      0
dtype: int64

Distribusi label awal:
final_label
negatif    1485
positif     390
netral       84
Name: count, dtype: int64


In [3]:
# ===== CLEANING (ringan, IndoBERTweet sudah terlatih di teks media sosial) =====
# KEEP : emoji, URL, @mention, #hashtag, tanda baca, huruf besar/kecil
# (IndoBERTweet punya normalisasi sendiri: user -> @USER, url -> HTTPURL, jadi tidak perlu dibuang)
# DO   : rapikan whitespace, buang komentar kosong, buang duplikat

def clean_text(t):
    t = str(t).replace("\u200b", "")          # zero-width space
    return re.sub(r"\s+", " ", t).strip()

d = df.copy()
n0 = len(d)

d = d[d["comment_text"].notna() & d["final_label"].notna()].copy()
d["comment_text"] = d["comment_text"].map(clean_text)
d = d[d["comment_text"] != ""]
n1 = len(d)

# Komentar sama dengan label BERBEDA = ambigu (cek, jangan diam-diam dibuang)
conf = d.groupby("comment_text")["final_label"].nunique()
conf = conf[conf > 1]

# Duplikat teks sama (apapun labelnya) harus dibuang sebelum split supaya tidak bocor ke test
d = d.drop_duplicates(subset=["comment_text"], keep="first").reset_index(drop=True)
n2 = len(d)

print(f"Awal                      : {n0}")
print(f"Setelah buang kosong/NaN  : {n1}  (-{n0-n1})")
print(f"Setelah buang duplikat    : {n2}  (-{n1-n2})")
print(f"Komentar sama tapi label beda (ambigu): {len(conf)}")
print("\nDistribusi label setelah cleaning:")
print(d["final_label"].value_counts())

Awal                      : 1959
Setelah buang kosong/NaN  : 1908  (-51)
Setelah buang duplikat    : 1867  (-41)
Komentar sama tapi label beda (ambigu): 0

Distribusi label setelah cleaning:
final_label
negatif    1446
positif     389
netral       32
Name: count, dtype: int64


In [4]:
# ===== STRATIFIED SPLIT =====
train, test = train_test_split(
    d, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=d["final_label"]
)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

summary = pd.DataFrame({
    "train": train["final_label"].value_counts(),
    "train_%": (train["final_label"].value_counts(normalize=True) * 100).round(1),
    "test": test["final_label"].value_counts(),
    "test_%": (test["final_label"].value_counts(normalize=True) * 100).round(1),
})
summary.loc["TOTAL"] = [len(train), 100.0, len(test), 100.0]
print(summary)

# Sanity: tidak boleh ada komentar yang sama di train dan test
overlap = set(train["comment_text"]) & set(test["comment_text"])
print("\nKomentar overlap train/test:", len(overlap))

              train  train_%   test  test_%
final_label                                
negatif      1156.0     77.4  290.0    77.5
positif       311.0     20.8   78.0    20.9
netral         26.0      1.7    6.0     1.6
TOTAL        1493.0    100.0  374.0   100.0

Komentar overlap train/test: 0


In [5]:
# ===== SAVE ke data/praprocessing/<nama_file>/ =====
train_path = OUTPUT_DIR / "train_raw.csv"
test_path = OUTPUT_DIR / "test_raw.csv"
train.to_csv(train_path, index=False, encoding="utf-8")
test.to_csv(test_path, index=False, encoding="utf-8")

print("Tersimpan:")
print(" ", train_path, f"({len(train)} baris)")
print(" ", test_path, f"({len(test)} baris)")

Tersimpan:
  C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data\praprocessing\labels_shafwa_hans_agreement\train_raw.csv (1493 baris)
  C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data\praprocessing\labels_shafwa_hans_agreement\test_raw.csv (374 baris)
